# 02 · QC and trimming: the decisions
**Goal:** check the raw reads, decide how to trim them, and show the evidence. **Input:** the FastQC reports from Euler (unpacked in notebook 01) and, for the adapter check, one `.sra` file. **Output:** none; this notebook records decisions. **Kernel:** Python 3 · **Time:** minutes.

Per run on Euler: `.sra` → FASTQ → **FastQC** (raw reads) → **fastp** trimming → kallisto, STAR, RSEM (notebook 01; [docs/pipeline.md](../docs/pipeline.md)). Terms: [docs/glossary.md](../docs/glossary.md).

In [1]:
# helpers.py: pipeline functions; runs = sample table
import sys; sys.path.insert(0, '../scripts')
import helpers as h
from IPython.display import display
runs = h.load_runs()

## 1 · Quality control of all reads
FastQC runs on **all** reads of every run (the `qc` stage on Euler). Its verdicts come from fixed thresholds (`limits.txt`); some WARNs are normal for RNA-seq.

| Check | WARN when | FAIL when | Expected |
|---|---|---|---|
| Per base sequence quality | lower quartile < 10 or median < 25 | < 5 / < 20 | PASS |
| Per sequence quality scores | most common mean quality < 27 | < 20 | PASS |
| **Adapter Content** | > 5 % of reads | > 10 % | PASS, else trim |
| Overrepresented sequences | one sequence > 0.1 % of reads | > 1 % | WARN common (Smart-seq2 primers) |
| Sequence Duplication Levels | < 70 % of reads remain after dedup | < 50 % | WARN normal (highly expressed genes) |

**Result:** base quality PASS in all 54 read files; **adapter content FAIL in all 54** (17–34 % of reads run into Nextera transposase sequence, because fragments are shorter than the 150-base read); overrepresented sequences and duplication WARN, as expected.

In [2]:
# FastQC verdicts of every read file (from the reports unpacked in notebook 01)
h.fastqc_summary(list(runs.index))

,Per base sequence quality,Per sequence quality scores,Adapter Content,Overrepresented sequences,Sequence Duplication Levels
SRR21355401_1,PASS,PASS,FAIL,WARN,WARN
SRR21355401_2,PASS,PASS,FAIL,WARN,WARN
SRR21355396_1,PASS,PASS,FAIL,WARN,WARN
SRR21355396_2,PASS,PASS,FAIL,WARN,WARN
SRR21355392_1,PASS,PASS,FAIL,WARN,WARN
SRR21355392_2,PASS,PASS,FAIL,WARN,WARN
SRR21355388_1,PASS,PASS,FAIL,WARN,WARN
SRR21355388_2,PASS,PASS,FAIL,WARN,WARN
SRR21355398_1,PASS,PASS,FAIL,WARN,WARN
SRR21355398_2,PASS,PASS,FAIL,WARN,WARN


## 2 · Trim or not?
Adapters fail in **every** sample, so **all** runs are trimmed with the same settings (trimming only some would create a technical difference between groups). fastp detects the adapter from the read pair, trims poly-G tails, keeps reads ≥ 20 bases and drops low-quality reads (about 4–6 % of pairs). Duplicates are **not** removed (see glossary).

**Effect of trimming** (early test on five runs first quantified untrimmed, vM25 reference): the pseudoaligned share rose by 2.1–2.7 points.

### Check: adapter and poly-G (SRR21355398, first 600,000 read pairs)
**Is it the adapter?** Yes.
- The 12 bases `CTGTCTCTTATA` start the Nextera adapter (expected for a Tn5 library) and occur in 27–30 % of reads, vs ≈ 5 expected by chance.
- The bases after them match the read's *own* mate's adapter in 96–99 % of cases, the other mate's in 0 %.
- After trimming, FastQC adapter content drops from 25–34 % to 0.0 % (two runs).

**Poly-G** (≥ 12 G in a row): the NovaSeq reads a cluster with no light as G. After a short insert (~49 bases) a read runs through the adapter (34), a sample barcode (8) and more library construct (~33), then out of template; the remaining ~24 cycles are dark:
```
read    [ insert ][ adapter ][ barcode ][ rest of construct ][ no template left: dark -> GGGGGGG ]
bases      ~49         34          8            ~33                  ~24
```
Only 4–6 % of reads have a poly-G run (27–30 % reach the adapter). The G's have high quality (mean Q32), so quality trimming misses them. Adapter cutting alone removes them in 91–93 % of these reads, fastp's filters drop 5–6 %, and 1.5–3 % keep a G run (about half real G-rich sequence). `--trim_poly_g` catches the rest, so **it is on for all 27 runs**. Without it (one run tested) pseudoalignment was identical (65.20 %) and counts changed by 0.05 %, but a few G-rich genes (Gm19220, Gm26822, Grip2) were inflated by residual poly-G reads (`results/qc/polyG_sensitivity_SRR21355398.csv`); none is differentially expressed.

In [3]:
# per mate: adapter evidence and the fate of poly-G reads (needs the .sra on this machine; re-runs fastq-dump and fastp on 600k pairs, ~1 min)
if h.has_sra('SRR21355398'):
    display(h.adapter_check('SRR21355398').T)
else:
    print('SRR21355398.sra is not on this machine: h.download_run(\'SRR21355398\') fetches it (needs sra-tools); the results are described in the text above')

mate,1,2
reads,600000.0,600000.0
adapter_start_%,27.2,29.6
expected_by_chance_reads,5.0,5.0
continues_own_adapter_%,95.8,98.5
matches_other_mate_adapter_%,0.0,0.0
polyG_reads_%,6.2,4.3
polyG_after_adapter_%,78.8,88.0
polyG_dropped_%,5.2,5.9
polyG_removed_%,93.3,91.1
polyG_remaining_%,1.5,3.0
